# Test incompressible flow inference with MuS-GNN models

In [ ]:
import torch
from torchvision import transforms
import numpy as np
import os
import graphs4cfd as gfd

Select device

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Inference

Load MuS-GNN models from [Lino et al. (2022)](https://doi.org/10.1063/5.0097679). These are:
- 1S-GNN: `1S-GNN-NsCircle-v1`
- 2S-GNN: `2S-GNN-NsCircle-v1`
- 3S-GNN: `3S-GNN-NsCircle-v1`
- 4S-GNN: `4S-GNN-NsCircle-v1`

In [ ]:
model1 = gfd.nn.NsOneScaleGNN  (model="1S-GNN-NsCircle-v1", device=device)
model2 = gfd.nn.NsTwoScaleGNN  (model="2S-GNN-NsCircle-v1", device=device)
model3 = gfd.nn.NsThreeScaleGNN(model="3S-GNN-NsCircle-v1", device=device)
model4 = gfd.nn.NsFourScaleGNN (model="4S-GNN-NsCircle-v1", device=device)

## They can also be loaded from a checkpoint file:
# directory = os.path.join("..", "..", "..", "graphs4cfd", "nn", "weights", "NsMuSGNN")
# model1 = gfd.nn.NsOneScaleGNN  (checkpoint=os.path.join(directory, "NsOneScaleGNN.chk"),   device=device)
# model2 = gfd.nn.NsTwoScaleGNN  (checkpoint=os.path.join(directory, "NsTwoScaleGNN.chk"),   device=device)
# model3 = gfd.nn.NsThreeScaleGNN(checkpoint=os.path.join(directory, "NsThreeScaleGNN.chk"), device=device)
# model4 = gfd.nn.NsFourScaleGNN (checkpoint=os.path.join(directory, "NsFourScaleGNN.chk"),  device=device)

print(f'Number of parameters in model1: {model1.num_params}')
print(f'Number of parameters in model2: {model2.num_params}')
print(f'Number of parameters in model3: {model3.num_params}')
print(f'Number of parameters in model4: {model4.num_params}')

Define the test datasets. These are available at [https://doi.org/10.5281/zenodo.7870707](https://doi.org/10.5281/zenodo.7870707).

In [ ]:
# NsCircle dataset (training dataset)
path = "<PATH TO DATASET NsCircle.h5>"
scaling = {'u': (-2.1,2.6), 'v': (-2.25,2.1), 'p': (-3.7,2.35), 'Re': (500,1000)} # This tuples are the (min, max) values for each variable in the training dataset
transform = transforms.Compose([
    gfd.transforms.ConnectKNN(6, period=(None,"auto")), # Detect the periodicity along the y-axis
    gfd.transforms.ScaleNs(scaling, 'uvp'),
    gfd.transforms.ScaleEdgeAttr(0.1),
    gfd.transforms.GridClustering([0.15, 0.30, 0.60]), # This builds the four-scale graph
])
ns_circle = gfd.datasets.NsCircle('uvp', path=path, transform=transform) # The format is 'uvp' because the dataset contains the velocity components and the pressure

# NsCircleMidRe dataset (testing dataset)
path = "<PATH TO DATASET NsCircleMidRe.h5>"
ns_circle_mid_re = gfd.datasets.NsCircle('uvp', path=path, transform=transform) # The format is 'uvp' because the dataset contains the velocity components and the pressure

# NsCircleLowRe dataset (Re-extrapolation dataset)
path = "<PATH TO DATASET NsCircleLowRe.h5>"
ns_circle_low_re = gfd.datasets.NsCircle('uvp', path=path, transform=transform) # The format is 'uvp' because the dataset contains the velocity components and the pressure

# NsCircleHighRe dataset (Re-extrapolation dataset)
path = "<PATH TO DATASET NsCircleHighRe.h5>"
ns_circle_high_re = gfd.datasets.NsCircle('uvp', path=path, transform=transform) # The format is 'uvp' because the dataset contains the velocity components and the pressure


# Print the number of samples/simulations in each dataset
print(f'Number of samples in ns_circle: {len(ns_circle)}')
print(f'Number of samples in ns_circle_mid_re: {len(ns_circle_mid_re)}')
print(f'Number of samples in ns_circle_low_re: {len(ns_circle_low_re)}')
print(f'Number of samples in ns_circle_high_re: {len(ns_circle_high_re)}')

It's inference time! We will use the test dataset to evaluate the models.

1. Inference on **NsCircle**

In [ ]:
sample = len(ns_circle)//2 # Idx of the sample/simulation from the dataset
n_out = 99 # Number of prediction steps
graph = ns_circle.get_sequence(sample, n_out=n_out)

# Inference with each model
pred1 = model1.solve(graph, n_out=n_out)
pred2 = model2.solve(graph, n_out=n_out)
pred3 = model3.solve(graph, n_out=n_out)
pred4 = model4.solve(graph, n_out=n_out)

# Compute the coefficients of determination (R^2) for each model
r21 = gfd.metrics.r2(pred1, graph.target)
r22 = gfd.metrics.r2(pred2, graph.target)
r23 = gfd.metrics.r2(pred3, graph.target)
r24 = gfd.metrics.r2(pred4, graph.target)
print(f'$R^2$ for model1: {r21}')
print(f'R^2 for model2: {r22}')
print(f'R^2 for model3: {r23}')
print(f'R^2 for model4: {r24}')
best_model = np.argmax([r21, r22, r23, r24])
print(f'The most accurate model is model{best_model+1}')

# Plot the ground truth and the predicted x-component of the velocity field for the best model
print(f'Plotting the ground truth and predicted x-component of the velocity field for model{best_model+1}')
pred = [pred1, pred2, pred3, pred4][best_model]
gfd.plot.compare_fields(graph.pos, graph.target[:,::3], pred[:,::3], bound=graph.bound)

1. Inference on **NsCircleMidRe**

In [ ]:
sample = len(ns_circle_mid_re)//2 # Idx of the sample/simulation from the dataset
n_out = 99 # Number of prediction steps
graph = ns_circle_mid_re.get_sequence(sample, n_out=n_out)

# Inference with each model
pred1 = model1.solve(graph, n_out=n_out)
pred2 = model2.solve(graph, n_out=n_out)
pred3 = model3.solve(graph, n_out=n_out)
pred4 = model4.solve(graph, n_out=n_out)

# Compute the coefficients of determination (R^2) for each model
r21 = gfd.metrics.r2(pred1, graph.target)
r22 = gfd.metrics.r2(pred2, graph.target)
r23 = gfd.metrics.r2(pred3, graph.target)
r24 = gfd.metrics.r2(pred4, graph.target)
print(f'$R^2$ for model1: {r21}')
print(f'R^2 for model2: {r22}')
print(f'R^2 for model3: {r23}')
print(f'R^2 for model4: {r24}')
best_model = np.argmax([r21, r22, r23, r24])
print(f'The most accurate model is model{best_model+1}')

# Plot the ground truth and the predicted x-component of the velocity field for the best model
print(f'Plotting the ground truth and predicted x-component of the velocity field for model{best_model+1}')
pred = [pred1, pred2, pred3, pred4][best_model]
gfd.plot.compare_fields(graph.pos, graph.target[:,::3], pred[:,::3], bound=graph.bound)

1. Inference on **NsCircleLowRe**

In [ ]:
sample = len(ns_circle_low_re)//2 # Idx of the sample/simulation from the dataset
n_out = 99 # Number of prediction steps
graph = ns_circle_low_re.get_sequence(sample, n_out=n_out)

# Inference with each model
pred1 = model1.solve(graph, n_out=n_out)
pred2 = model2.solve(graph, n_out=n_out)
pred3 = model3.solve(graph, n_out=n_out)
pred4 = model4.solve(graph, n_out=n_out)

# Compute the coefficients of determination (R^2) for each model
r21 = gfd.metrics.r2(pred1, graph.target)
r22 = gfd.metrics.r2(pred2, graph.target)
r23 = gfd.metrics.r2(pred3, graph.target)
r24 = gfd.metrics.r2(pred4, graph.target)
print(f'$R^2$ for model1: {r21}')
print(f'R^2 for model2: {r22}')
print(f'R^2 for model3: {r23}')
print(f'R^2 for model4: {r24}')
best_model = np.argmax([r21, r22, r23, r24])
print(f'The most accurate model is model{best_model+1}')

# Plot the ground truth and the predicted x-component of the velocity field for the best model
print(f'Plotting the ground truth and predicted x-component of the velocity field for model{best_model+1}')
pred = [pred1, pred2, pred3, pred4][best_model]
gfd.plot.compare_fields(graph.pos, graph.target[:,::3], pred[:,::3], bound=graph.bound)

1. Inference on **NsCircleHighRe**

In [ ]:
sample = len(ns_circle_high_re)//2 # Idx of the sample/simulation from the dataset
n_out = 99 # Number of prediction steps
graph = ns_circle_high_re.get_sequence(sample, n_out=n_out)

# Inference with each model
pred1 = model1.solve(graph, n_out=n_out)
pred2 = model2.solve(graph, n_out=n_out)
pred3 = model3.solve(graph, n_out=n_out)
pred4 = model4.solve(graph, n_out=n_out)

# Compute the coefficients of determination (R^2) for each model
r21 = gfd.metrics.r2(pred1, graph.target)
r22 = gfd.metrics.r2(pred2, graph.target)
r23 = gfd.metrics.r2(pred3, graph.target)
r24 = gfd.metrics.r2(pred4, graph.target)
print(f'$R^2$ for model1: {r21}')
print(f'R^2 for model2: {r22}')
print(f'R^2 for model3: {r23}')
print(f'R^2 for model4: {r24}')
best_model = np.argmax([r21, r22, r23, r24])
print(f'The most accurate model is model{best_model+1}')

# Plot the ground truth and the predicted x-component of the velocity field for the best model
print(f'Plotting the ground truth and predicted x-component of the velocity field for model{best_model+1}')
pred = [pred1, pred2, pred3, pred4][best_model]
gfd.plot.compare_fields(graph.pos, graph.target[:,::3], pred[:,::3], bound=graph.bound)